# Track 2 Codec Root-Cause Analysis

This notebook summarizes the repaired Track 2 baseline and compares `raw/basic`, `Stage 0 baseline`, `Stage 0 enhanced`, `Stage 1 baseline`, and `Stage 1 enhanced`.

Outputs are generated by `src/scripts/analyze_track2_codec_root_cause.py` and saved to:
- `logs/track2_codec_root_cause_summary.csv`
- `logs/track2_codec_root_cause_figs/`


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPO_ROOT = Path('/home/018219422/lidar_pointcloud_compression')
SUMMARY_CSV = REPO_ROOT / 'logs/track2_codec_root_cause_summary.csv'
FIG_DIR = REPO_ROOT / 'logs/track2_codec_root_cause_figs'
SCRIPT = REPO_ROOT / 'src/scripts/analyze_track2_codec_root_cause.py'

if not SUMMARY_CSV.exists():
    subprocess.run(['python', str(SCRIPT)], check=True, cwd=str(REPO_ROOT))

df = pd.read_csv(SUMMARY_CSV)
df.head()


In [ ]:
agg = df.groupby('setting').agg(
    valid_mask_iou=('valid_mask_iou', 'mean'),
    range_mae_valid=('range_mae_valid', 'mean'),
    row_profile_mae=('row_profile_mae', 'mean'),
    banding_score=('banding_score', 'mean'),
    grad_row_mae=('grad_row_mae', 'mean'),
    grad_col_mae=('grad_col_mae', 'mean'),
    high_frequency_score=('high_frequency_score', 'mean'),
    pred_count=('pred_count', 'mean'),
    oracle_recall3d_03=('oracle_recall3d_03', 'mean'),
    mean_best_iou3d_03=('mean_best_iou3d_03', 'mean'),
).sort_index()
agg


In [ ]:
driver = pd.crosstab(df['setting'], df['dominant_failure_driver'])
driver


In [ ]:
worst = (
    df[df['setting'] == 'stage0_baseline']
      .sort_values(['oracle_recall3d_03', 'valid_mask_iou', 'row_profile_mae'], ascending=[True, True, False])
      .head(12)
)
worst[['sample_id', 'valid_mask_iou', 'range_mae_valid', 'row_profile_mae', 'banding_score', 'grad_row_mae', 'grad_col_mae', 'pred_count', 'oracle_recall3d_03', 'mean_best_iou3d_03', 'dominant_failure_driver']]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for ax, metric in zip(axes, ['valid_mask_iou', 'row_profile_mae', 'oracle_recall3d_03']):
    pivot = df.pivot(columns='setting', values=metric)
    pivot.boxplot(ax=ax)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=25)
plt.show()


In [ ]:
for name in sorted(FIG_DIR.glob('*_worst.png')):
    print(name.name)
    display(Image(filename=str(name)))
for name in sorted(FIG_DIR.glob('*_median.png')):
    print(name.name)
    display(Image(filename=str(name)))
